# Cleaning the Data

## Init

In [0]:
from pyspark.sql.functions import regexp_replace, split, regexp_extract, col
from functools import reduce

## Reading from Bronze Schema

In [0]:
total_pitch_df = spark.table('pitch_data_2025.bronze.total_pitch_data')
attack_zone_df = spark.table('pitch_data_2025.bronze.attack_zone_data')
pybaseball_df = spark.table('pitch_data_2025.bronze.pybaseball_data')

## Filter Columns

In [0]:
# Columns to keep
tp_col = ['Date', 'Player_ID', 'Pitcher', 'Pitch', 'MPH', 'Spin_Rate', 'Zone', 'Count', 'Inning', 'Total_Pitch_Count']
az_py_col = ['game_date', 'pitcher', 'player_name', 'pitch_name', 'release_speed', 'release_spin_rate', 'zone', 'balls', 'strikes',
             'inning', 'inning_topbot', 'at_bat_number', 'pitch_number']
az_col = ['attack_zone', 'events', 'description', 'age_pit_legacy', 'p_throws', 'pitch_type']
py_col = ['pfx_x', 'pfx_z', 'plate_x', 'plate_z']

# selecting the columns
total_pitch_df = total_pitch_df.select(*tp_col)
attack_zone_df = attack_zone_df.select(*az_py_col, *az_col)
pybaseball_df = pybaseball_df.select(*az_py_col, *py_col)

## Total Pitching Cleaning

In [0]:
# Inning
total_pitch_df = total_pitch_df.withColumnRenamed('Inning', 'Inning_og')

total_pitch_df = (
    total_pitch_df
    .withColumn('inning_topbot', regexp_extract('Inning_og', r'(Top|Bot)', 1))
    .withColumn('inning', regexp_extract('Inning_og', r"(\d+)", 1).cast('int'))
)

## Removing Inning_og
total_pitch_df = total_pitch_df.drop('Inning_og')

# Pitcher Name
total_pitch_df = (
    total_pitch_df
    .withColumn('Pitcher', regexp_replace('Pitcher', r'\([RL]\)', ''))
)

invalid_names = total_pitch_df.filter(total_pitch_df.Pitcher.rlike(r'[()]'))

assert invalid_names.count() == 0, ("Error: Parentheses names found")

# Count
total_pitch_df = (
    total_pitch_df
    .withColumn('balls', regexp_extract('Count', r'(\d)-', 1).try_cast('int'))
    .withColumn('strikes', regexp_extract('Count', r'-(\d)', 1).try_cast('int'))
)

## Removing Count
total_pitch_df = total_pitch_df.drop('Count')

# Changing Pitch Names
pitch_map = {
    '4-Seam':'4-Seam Fastball',
    'Curve':'Curveball',
    'Change':'Changeup',
    'Split':'Split-Finger',
    'Kn. Curve':'Knuckle Curve',
    'Knuckle':'Knuckleball',
    'UN':'Unknown'
}

total_pitch_df = total_pitch_df.replace(to_replace=pitch_map, subset=['Pitch'])

# Renaming columns
col_map = {
    'Date':'game_date',
    'Player_ID':'pitcher',
    'Pitcher':'player_name',
    'Pitch':'pitch_name',
    'MPH':'release_speed',
    'Spin_Rate':'release_spin_rate',
    'Zone':'zone',
    'Total_Pitch_Count': 'total_pitch_count'
}

total_pitch_df = total_pitch_df.toDF(*[col_map.get(c, c) for c in total_pitch_df.columns])

## Handling Null Values

In [0]:
# Get Nulls Function
def get_nulls(df):
    nulls = {}
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        if null_count > 0:
            nulls[c] = null_count
    return nulls

# Checking which columns in which dfs have NA values
null_columns = {
    'total_pitch_df': get_nulls(total_pitch_df),
    'attack_zone_df': get_nulls(attack_zone_df),
    'pybaseball_df': get_nulls(pybaseball_df)
}

print(null_columns)
# total_pitch: Null values appear in spin_rate and release_speed
# attack_zone: spin_rate, release_speed, and events
# pybaseball: spin_rate, release_speed, pfx_x, and pfx_z


# Filling null values
## Total pitch: replacing speed and spin rate with 0 values, necissary for merging in gold layer
total_pitch_df = total_pitch_df.na.fill({'release_speed': 0, 'release_spin_rate': 0})
## Attack zone: replacing event values with 'no_field_event'
attack_zone_df = attack_zone_df.na.fill({'release_speed': 0, 'release_spin_rate': 0,
                                         'events': 'no_field_event'})
## Pybaseball: keeping pfx_x and pfx_z as null values
pybaseball_df = pybaseball_df.na.fill({'release_speed': 0, 'release_spin_rate': 0})


# Testing there are no null values in total pitch and attack zone
assert get_nulls(total_pitch_df) == {}, ("Error: Null values found in Total Pitch")
assert get_nulls(attack_zone_df) == {}, ("Error: Null values found in Attack Zone")
pyb_nulls = get_nulls(pybaseball_df)
assert "release_speed" not in pyb_nulls and "release_spin_rate" not in pyb_nulls, ("Error: Null values found in Pybaseball")

## Changing Column Types

In [0]:
total_pitch_df = (
    total_pitch_df
    .withColumn("pitcher", col("pitcher").cast("bigint"))
    .withColumn("zone", col("zone").cast("bigint"))
    .withColumn("release_spin_rate", col("release_spin_rate").cast("bigint"))
)

attack_zone_df = attack_zone_df.withColumn("release_spin_rate", col("release_spin_rate").cast("bigint"))

pybaseball_df = (
    pybaseball_df
    .withColumn("zone", col("zone").cast("bigint"))
    .withColumn("release_spin_rate", col("release_spin_rate").cast("bigint"))
)

## Writing into Silver Schema

In [0]:
total_pitch_df.write.mode("overwrite").saveAsTable("pitch_data_2025.silver.total_pitch_data")
attack_zone_df.write.mode("overwrite").saveAsTable("pitch_data_2025.silver.attack_zone_data")
pybaseball_df.write.mode("overwrite").saveAsTable("pitch_data_2025.silver.pybaseball_data")